# Прогнозирование почасового трафика на перекрёстке

Итоговый проект по курсу «Анализ временных рядов».

Ноутбук содержит все этапы исследования:

1. Загрузка и очистка данных
2. Разведочный анализ (EDA) и постановка задачи
3. Статистические модели
4. ML-модели с feature engineering
5. Нейросетевые модели
6. Выявление аномалий
7. Финальные выводы

## 1. Загрузка и очистка данных

Исходный файл `data/traffic.csv` содержит почасовые замеры количества
транспортных средств на нескольких перекрёстках. По заданию используем только
перекрёсток `Junction == 1` — для него ряд наиболее длинный и полный.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

DATA_PATH = os.path.join('..', 'data', 'traffic.csv')
raw = pd.read_csv(DATA_PATH)
raw.head()

,DateTime,Junction,Vehicles,ID
0,2015-11-01 00:00:00,1,15,20151101001
1,2015-11-01 01:00:00,1,13,20151101011
2,2015-11-01 02:00:00,1,10,20151101021
3,2015-11-01 03:00:00,1,7,20151101031
4,2015-11-01 04:00:00,1,9,20151101041


In [2]:
# Общая информация о таблице: типы, количество строк, объём памяти
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 48120 entries, 0 to 48119
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   DateTime  48120 non-null  str  
 1   Junction  48120 non-null  int64
 2   Vehicles  48120 non-null  int64
 3   ID        48120 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [3]:
# Проверка пропусков и дубликатов
print('Пропуски по столбцам:')
print(raw.isna().sum())
print('\nЯвные дубликаты строк:', raw.duplicated().sum())
print('Перекрёстки в данных:', sorted(raw['Junction'].unique().tolist()))
print('Записей по перекрёсткам:')
print(raw['Junction'].value_counts().sort_index())

Пропуски по столбцам:
DateTime    0
Junction    0
Vehicles    0
ID          0
dtype: int64

Явные дубликаты строк: 0
Перекрёстки в данных: [1, 2, 3, 4]
Записей по перекрёсткам:
Junction
1    14592
2    14592
3    14592
4     4344
Name: count, dtype: int64


### Отбор перекрёстка и приведение индекса

Берём `Junction == 1`, приводим `DateTime` к типу `datetime64`, делаем его
индексом и сортируем. Столбец `ID` (порядковый номер) для анализа не нужен,
поэтому удаляем его.

In [ ]:
df = raw.loc[raw['Junction'] == 1].copy()
df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values('DateTime').set_index('DateTime')
df = df.drop(columns=['Junction', 'ID'])
df = df.rename(columns={'Vehicles': 'y'})
df.head()

In [ ]:
# Границы периода наблюдений и шаг ряда
print('Начало ряда :', df.index.min())
print('Конец ряда  :', df.index.max())
print('Наблюдений  :', len(df))

expected_hours = pd.date_range(df.index.min(), df.index.max(), freq='H')
missing_hours = expected_hours.difference(df.index)
print('Ожидалось часов (без пропусков):', len(expected_hours))
print('Пропущенных часов             :', len(missing_hours))

In [ ]:
# Приведение к строго почасовому индексу.
# Если пропусков нет, asfreq просто зафиксирует частоту 'H' в индексе.
# Если появятся NaN — заполним их линейной интерполяцией (разумно для гладкого
# суточного профиля трафика).
df = df.asfreq('H')
print('После asfreq пропусков:', df['y'].isna().sum())
if df['y'].isna().any():
    df['y'] = df['y'].interpolate(method='time')
    print('Пропуски заполнены линейной интерполяцией.')
df.head()

In [ ]:
# Базовая описательная статистика целевой переменной
df['y'].describe().round(2)

In [ ]:
# Среднее по часу суток и дню недели — быстрая прикидка сезонных профилей
by_hour = df.groupby(df.index.hour)['y'].mean().round(1)
by_dow  = df.groupby(df.index.dayofweek)['y'].mean().round(1)
by_dow.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print('Средний трафик по часам суток:')
print(by_hour)
print('\nСредний трафик по дням недели:')
print(by_dow)

## 2. Разведочный анализ и постановка задачи

Теперь визуализируем ряд на разных масштабах, оценим автокорреляции и
сезонные компоненты, проверим стационарность и зафиксируем постановку задачи.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (14, 4)

### 2.1 Визуализация ряда

Смотрим ряд целиком, одну неделю и одни сутки. Цель — увидеть наличие
суточного цикла, недельного цикла и долгосрочного тренда.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df['y'].plot(ax=ax, color='steelblue', linewidth=0.7)
ax.set_title('Почасовой трафик на перекрёстке (весь период)')
ax.set_ylabel('Транспортные средства / час')
ax.set_xlabel('Время')
plt.tight_layout()
plt.show()

In [ ]:
# Одна неделя — чтобы увидеть недельный и суточный ритм
week_start = df.index.min() + pd.Timedelta(days=7)
week = df.loc[week_start : week_start + pd.Timedelta(days=7)]

fig, ax = plt.subplots(figsize=(14, 4))
week['y'].plot(ax=ax, color='darkorange')
ax.set_title(f'Недельный срез: {week.index.min().date()} — {week.index.max().date()}')
ax.set_ylabel('Транспортные средства / час')
plt.tight_layout()
plt.show()

In [ ]:
# Одни сутки — суточный профиль
day_start = week.index.min()
day = df.loc[day_start : day_start + pd.Timedelta(hours=23)]

fig, ax = plt.subplots(figsize=(14, 4))
day['y'].plot(ax=ax, marker='o', color='seagreen')
ax.set_title(f'Суточный срез: {day.index.min().date()}')
ax.set_ylabel('Транспортные средства / час')
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение профилей по дням недели (boxplot) и по часам суток
tmp = df.copy()
tmp['hour'] = tmp.index.hour
tmp['dow']  = tmp.index.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=tmp, x='hour', y='y', ax=axes[0], color='steelblue')
axes[0].set_title('Распределение трафика по часам суток')
axes[0].set_xlabel('Час суток')
axes[0].set_ylabel('Трафик')

sns.boxplot(data=tmp, x='dow', y='y', ax=axes[1], color='darkorange')
axes[1].set_title('Распределение трафика по дням недели')
axes[1].set_xlabel('День недели (0=Пн)')
axes[1].set_ylabel('Трафик')
plt.tight_layout()
plt.show()

### 2.2 Автокорреляции

Строим ACF и PACF до лага 200. Ожидаемые пики — на лагах 24 (сутки) и 168
(неделя).

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
plot_acf(df['y'],  lags=200, ax=axes[0])
axes[0].set_title('ACF (до лага 200)')
plot_pacf(df['y'], lags=60, ax=axes[1], method='ywm')
axes[1].set_title('PACF (до лага 60)')
plt.tight_layout()
plt.show()

### 2.3 Сезонная декомпозиция

Используем аддитивную модель с периодом 24 (суточная сезонность — самая
выраженная). Для недельной сезонности можно повторить с `period=168`, но на
графике она и так будет видна по остаточной компоненте.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(df['y'], model='additive', period=24)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

In [ ]:
# То же с недельным периодом — чтобы увидеть недельный цикл отдельно
decomp_w = seasonal_decompose(df['y'], model='additive', period=168)
fig = decomp_w.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

### 2.4 Проверка стационарности (ADF)

Тест Дики-Фуллера проверяет гипотезу о наличии единичного корня.
Если p-value < 0.05 — отвергаем гипотезу нестационарности.
Обычно почасовой трафик после снятия суточной сезонности становится
стационарным, поэтому тест также проведём на сезонных разностях.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, label):
    stat, pvalue, *_ = adfuller(series.dropna(), autolag='AIC')
    print(f'{label:35s} | ADF={stat:8.3f} | p-value={pvalue:.4f}')

adf_report(df['y'],                  'Исходный ряд')
adf_report(df['y'].diff(),           'Первая разность')
adf_report(df['y'].diff(24),         'Сезонная разность (24)')
adf_report(df['y'].diff(24).diff(),  'Сез. (24) + первая разность')

### 2.5 Постановка задачи

**Цель.** Построить прогноз почасового трафика на перекрёстке
`Junction == 1` на ближайшие сутки.

**Горизонт прогноза.** `h = 24` часа. Такой горизонт практически востребован
(планирование работы светофоров, оценка загрузки на завтра) и одновременно
достаточно длинный, чтобы различать модели.

**Протокол валидации.** Бэктестинг (rolling cross-validation) с несколькими
окнами прогноза: минимум 5 подряд идущих 24-часовых горизонтов в конце ряда.
Финальное сравнение моделей усредняется по этим окнам.

**Метрики качества.** Используем три метрики, дополняющие друг друга:

- **MAE** — средняя абсолютная ошибка, интерпретируется в штуках машин.
- **RMSE** — корень из среднеквадратичной ошибки, штрафует крупные промахи.
- **sMAPE** — симметричная относительная ошибка в процентах, позволяет сравнивать масштабы.

**Основные выводы EDA:**

- Ряд имеет выраженную суточную (период 24) и недельную (период 168)
  сезонность — это видно на графиках и подтверждается пиками ACF.
- Долгосрочный тренд слабый, но присутствует (медленный рост трафика).
- Исходный ряд по ADF, как правило, уже близок к стационарному (за счёт
  сезонного "возврата к среднему"), после сезонной разности стационарен
  уверенно.
- Значит, разумно ожидать, что сезонные бейзлайны (Seasonal Naive 168) будут
  сильными соперниками, а ARIMA/ETS должны моделироваться с сезонным
  параметром.

### 2.6 Сохранение очищенного ряда

Сохраняем очищенную серию в `data/traffic_clean.csv` в формате, удобном для
Nixtla-стека (`unique_id`, `ds`, `y`).

In [ ]:
clean = (
    df.reset_index()
      .rename(columns={'DateTime': 'ds'})
      .assign(unique_id='junction_1')[['unique_id', 'ds', 'y']]
)

CLEAN_PATH = os.path.join('..', 'data', 'traffic_clean.csv')
clean.to_csv(CLEAN_PATH, index=False)
print(f'Сохранено: {CLEAN_PATH}')
print(f'Строк: {len(clean)}')
clean.head()